## Chapter 04: Prompt Engineering

### Principles of prompt engineering

#### 1. Clear instructions

In [2]:
import os
from openai import OpenAI

from config import DEEPSEEK_API_KEY

In [3]:
client = OpenAI(
  api_key=DEEPSEEK_API_KEY,
  base_url=("https://api.deepseek.com")
)

In [ ]:
# Set the metaprompt i.e. define how the model should behave
system_message = """
You are an AI assistant that helps humans by generating tutorials given a text.
You will be provided with a text. If the text contains any kind of instructions 
on how to proceed with something, generate a tutorial in a bullet list.
Otherwise, inform the user that the text does not contain any instructions.

Text:
"""

# Set the query i.e. where the user will ask the model its questions
instructions = """
To prepare the known sauce from Genova, Italy, you can start by toasting the pine 
nuts to then coarsely chop them in a kitchen mortar together with basil and garlic.
Then, add half of the oil in the kitchen mortar and season with salt and pepper.
Finally, transfer the pesto to a bowl and stir the grated Parmesan cheese.
"""

In [8]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": instructions},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [7]:
print(
  response.choices[0].message.content
)

Here's a tutorial on how to prepare the sauce:

- **Toast the pine nuts**: Place the pine nuts in a dry pan over medium heat and toast them lightly, stirring frequently, until golden and fragrant. Let them cool.
- **Crush the ingredients**: In a kitchen mortar, combine the toasted pine nuts, fresh basil leaves, and peeled garlic cloves. Coarsely crush and grind them with the pestle until a paste forms.
- **Incorporate oil and season**: While continuing to mix, gradually pour in half of the olive oil. Season with salt and pepper to taste, blending until well combined.
- **Finish with cheese**: Transfer the pesto from the mortar to a bowl. Add the grated Parmesan cheese and stir thoroughly until the cheese is fully incorporated. Your pesto is ready to use.


In [9]:
# Pass the model another text that doesn't contain any instructions
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": "The sun is shining and dogs are running on the beach."},
	]
)

In [10]:
print(
  response.choices[0].message.content
)

The text does not contain any instructions. It is a descriptive statement about a sunny beach scene with dogs.


#### 2. Split complex tasks into subtasks

In [ ]:
system_message = """
You are an AI assistant that summarizes articles.
To complete this task, do the following subtasks:

Read the provided article context comprehensively and identify the main topic and 
key points
Generate a paragraph summary of the current article context that captures the 
essential information and conveys the main idea
Print each step of the process

Article:
"""

article = """
Recurrent neural networks, long short-term memory and gated recurrent neural networks 
in particular, have been firmly established as state of the art approaches in sequence 
modeling and transduction problems such as large language modeling and machine translation. 
Numerous efforts have since continued to push the boundaries of recurrent language 
models and encoder-decoder architectures.
Recurrent models typically factor computation along the symbol positions of the 
input and output sequences. Aligning the positions to steps in computation time, 
they generate a sequence of hidden states ht, as a function of the previous hidden 
state ht-1 and the input for position t. This inherently sequential nature precludes 
parallelization within training examples, which becomes critical at longer sequence 
lengths, as memory constraints limit batching across examples. Recent work has 
achieved significant improvements in computational efficiency through factorization 
tricks and conditional computation, while also improving model performance in case 
of the latter. The fundamental constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling 
and transduction models in various tasks, allowing modeling of dependencies without 
regard to their distance in the input or output sequences. In all but a few cases, 
however, such attention mechanisms are used in conjuction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence 
and instead relying entirely on an attention mechanism to draw global dependencies 
between input and output.
The transformer allows for significantly more parallelization and can reach a new 
state of the art in translation quality after being trained for as little as twelve 
hours on eight P100 GPUs.
"""

In [12]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages= [
    {"role": "system", "content": system_message},
    {"role": "user", "content": article},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [13]:
print(
  response.choices[0].message.content
)

Step 1: Read and analyze the article context. The article discusses the limitations of recurrent neural networks (RNNs), particularly their sequential computation which hinders parallelization and efficiency for long sequences. It notes that attention mechanisms are typically combined with RNNs to capture dependencies. The key point is the proposal of the Transformer architecture, which eliminates recurrence and relies solely on attention to model global dependencies, resulting in significantly more parallelization and state-of-the-art translation performance.

Step 2: Generate a summary paragraph that conveys the essential information. The summary should state the problem with recurrent models, the existing use of attention, and the novel Transformer approach and its benefits.

Summary: Recurrent neural networks, while dominant in sequence modeling and transduction tasks, are inherently limited by their sequential computation, which restricts parallelization and efficiency for long se